# 03 — Gradient-only READ and hard idle controls

This notebook consumes only the sanitized clean manifest and frozen J-Lens directions. It computes READ_IG, READ_local, and the labelled static-capacity baseline, then constructs and scores the answer-type-matched idle controls. It does not import intervention code or read notebook 02's outputs; hard-control READ is frozen before hard-control causal truth exists.

**Outputs:** `artifacts/final/03_cheap.json`, `03_hard_manifest.json`, and `03_hard_cheap.json`.

In [1]:
from __future__ import annotations

import ast
import hashlib
import json
import os
import sys
from collections import Counter
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
ARTIFACTS = ROOT / 'artifacts' / 'final'

from src.cheap_read import (
    compute_base_cheap_read,
    compute_hard_dashboard_read,
    summarize_cheap_read_artifact,
)
from src.datasets import (
    HARD_DASHBOARD_TEMPLATES,
    build_hard_dashboard_candidates,
    validate_sanitized_manifest,
    verify_hard_dashboard_candidates,
)
from src.jlens_interface import load_model, release_model

def load_json(path: Path) -> dict:
    return json.loads(path.read_text())

def write_json(path: Path, value: object) -> None:
    path.write_text(json.dumps(value, indent=2, sort_keys=True) + '\n')

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def progress(done: int, total: int, row: dict) -> None:
    if done == 1 or done == total or done % 10 == 0:
        print(f'{done}/{total}: {row["pair_id"]}')

cheap_source = ROOT / 'src' / 'cheap_read.py'
tree = ast.parse(cheap_source.read_text())
imports = []
for node in ast.walk(tree):
    if isinstance(node, ast.Import):
        imports.extend(alias.name for alias in node.names)
    elif isinstance(node, ast.ImportFrom) and node.module:
        imports.append(node.module)
forbidden = [name for name in imports if any(term in name for term in ('causal_read', 'intervention', 'patching'))]
if forbidden:
    raise RuntimeError(f'Cheap READ firewall violation: {forbidden}')
print('Static cheap-READ import firewall: PASS', sorted(imports))

clean_path = ARTIFACTS / '01_clean_manifest.json'
direction_path = ARTIFACTS / '01_directions.pt'
clean = validate_sanitized_manifest(load_json(clean_path))
direction_cache = torch.load(direction_path, map_location='cpu', weights_only=False)
clean_sha = sha256(clean_path)
direction_sha = sha256(direction_path)

bundle = load_model()
try:
    base_cheap = compute_base_cheap_read(
        bundle,
        clean,
        direction_cache,
        clean_manifest_sha256=clean_sha,
        direction_cache_sha256=direction_sha,
        cheap_read_sha256=sha256(cheap_source),
        ig_steps=16,
        progress=progress,
    )
    base_cheap_path = ARTIFACTS / '03_cheap.json'
    write_json(base_cheap_path, base_cheap)

    hard_candidates = build_hard_dashboard_candidates(clean['rows'], bundle.tokenizer)
    hard_rows = verify_hard_dashboard_candidates(
        bundle.hf_model, bundle.tokenizer, hard_candidates
    )
    status_counts = Counter(row['verification_status'] for row in hard_rows)
    reason_counts = Counter(reason for row in hard_rows for reason in row['verification_reasons'])
    verified_groups = {row['dependency_group'] for row in hard_rows if row['verification_status'] == 'VERIFIED_HARD'}
    hard_manifest = {
        'schema_version': 'read-stress-v6-hard-dashboard-manifest-v1',
        'seed': 1729,
        'model': clean['model'],
        'selection': clean['selection'],
        'source_clean_manifest': {
            'path': str(clean_path.relative_to(ROOT)),
            'sha256': clean_sha,
            'protocol_sha256': clean['protocol_sha256'],
        },
        'direction_cache': {
            'path': str(direction_path.relative_to(ROOT)),
            'bytes': direction_path.stat().st_size,
            'sha256': direction_sha,
        },
        'design': {
            'control_kind': 'fixed calibration-anchor fact with matched semantic relation and answer class',
            'concept_irrelevance_by_construction': True,
            'exact_source_context_prefix_required': True,
            'template_selection_used_heldout_outcomes': False,
            'numeric_example_not_applicable': 'Frozen engines output element symbols or capital-city names, never numbers.',
            'templates': HARD_DASHBOARD_TEMPLATES,
        },
        'verification_contract': {
            'correctness': 'fixed hard target is clean top-1 in both prompts',
            'written': 'frozen explicit-concept WRITTEN score clears the frozen threshold',
            'written_provenance': 'exact prefix through the concept token is unchanged',
            'failed_rows_excluded_not_relabeled': True,
        },
        'counts': {
            'candidates': len(hard_rows),
            'verified_hard': status_counts['VERIFIED_HARD'],
            'unverified_hard': status_counts['UNVERIFIED_HARD'],
            'dependency_groups_verified': len(verified_groups),
            'reason_counts': dict(reason_counts),
        },
        'rows': hard_rows,
        'causal_interchange_outputs_included': False,
        'edited_metrics_included': False,
    }
    hard_manifest_path = ARTIFACTS / '03_hard_manifest.json'
    write_json(hard_manifest_path, hard_manifest)
    hard_cheap = compute_hard_dashboard_read(
        bundle,
        hard_manifest,
        direction_cache,
        hard_manifest_path=str(hard_manifest_path.relative_to(ROOT)),
        hard_manifest_sha256=sha256(hard_manifest_path),
        direction_cache_sha256=direction_sha,
        cheap_read_sha256=sha256(cheap_source),
        ig_steps=16,
        progress=progress,
    )
    hard_cheap_path = ARTIFACTS / '03_hard_cheap.json'
    write_json(hard_cheap_path, hard_cheap)
finally:
    del bundle
    release_model()

print(json.dumps({
    'firewall': 'PASS',
    'base': summarize_cheap_read_artifact(base_cheap),
    'hard': summarize_cheap_read_artifact(hard_cheap),
    'hard_counts': hard_manifest['counts'],
}, indent=2))


Static cheap-READ import firewall: PASS ['__future__', 'collections.abc', 'torch', 'torch', 'typing']


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

1/77: symmetric-000


10/77: symmetric-015


20/77: symmetric-025


...

50/77: symmetric-073


60/77: symmetric-084


70/77: symmetric-098


77/77: symmetric-105


{
  "firewall": "PASS",
  "base": {
    "schema_version": "symmetric-cheap-read-v1",
    "rows": 77,
    "selected_layer": 16,
    "ig_steps": 16,
    "engine_READ_IG_median": 0.21024686098098755,
    "dashboard_READ_IG_median": 0.005552652291953564,
    "causal_outputs_consumed": false
  },
  "hard": {
    "schema_version": "read-stress-v6-hard-dashboard-cheap-v1",
    "rows": 77,
    "selected_layer": 16,
    "ig_steps": 16,
    "hard_dashboard_READ_IG_median": 0.00631862273439765,
    "causal_outputs_consumed": false
  },
  "hard_counts": {
    "candidates": 77,
    "verified_hard": 77,
    "unverified_hard": 0,
    "dependency_groups_verified": 24,
    "reason_counts": {}
  }
}
